# Fixing Links Notebook

Before running this notebook, make sure to run the following command in the terminal to install the required packages:

```bash
bundle install
make all
ruby parse_htmlproofer_log.rb 
```

Each command should be run separately and the final two commands create files for all the htmlproofer errors and warnings. This notebook loads the final csv file to help you see what links exists. You will also need to install the `pandas` library if you haven't already. You can do this by running:

```bash
pip install pandas
```

## Load Libraries and Data

In [46]:
import pandas as pd

In [59]:
df = pd.read_csv("htmlproofer-report.csv")
# Lower case the column names
df.columns = df.columns.str.lower()
print(f"Number of errors: {len(df)}")

Number of errors: 212


In [60]:
message_counts = df.message.value_counts().reset_index()
print(f"Number of unique error messages: {len(message_counts)}")
message_counts[(message_counts['count']>1)]

Number of unique error messages: 150


,message,count
0,http://www.python.org/ is not an HTTPS link,6
1,http://niche-canada.org/2018/03/23/a-decade-of...,4
2,http://clionauta.hypotheses.org/16979 is not a...,4
3,External link https://github.com/orgs/programm...,4
4,External link https://central.github.com/mac/l...,4
5,internally linking to nocoes-basicas-paginas-w...,4
6,image /images/website/index/woman-using-tabula...,3
7,http://www.loc.gov/maps/collections is not an ...,3
8,http://programminghistorian.org/ is not an HTT...,3
9,http://www.crummy.com/software/BeautifulSoup/ ...,3


In [61]:
file_counts = df.file.value_counts().reset_index()
print(f"Number of unique files with errors: {len(file_counts)}")
file_counts[file_counts['count']>1]

Number of unique files with errors: 123


,file,count
0,_site/pt/licoes/introducao-instalacao-python/i...,6
1,_site/en/lessons/collaborative-blog-with-jekyl...,5
2,_site/en/lessons/retired/intro-to-augmented-re...,4
3,_site/en/contribute/index.html,4
4,_site/en/lessons/interactive-data-visualizatio...,4
5,_site/en/lessons/retired/graph-databases-and-S...,4
6,_site/pt/index.html,3
7,_site/en/lessons/applied-archival-downloading-...,3
8,_site/en/lessons/detecting-text-reuse-with-pas...,3
9,_site/en/lessons/exploring-and-analyzing-netwo...,3


In [62]:
file_counts_df = df.file.value_counts().reset_index()
file_counts_df['count_index'] = file_counts_df.index

file_counts_df

,file,count,count_index
0,_site/pt/licoes/introducao-instalacao-python/i...,6,0
1,_site/en/lessons/collaborative-blog-with-jekyl...,5,1
2,_site/en/lessons/retired/intro-to-augmented-re...,4,2
3,_site/en/contribute/index.html,4,3
4,_site/en/lessons/interactive-data-visualizatio...,4,4
...,...,...,...
118,_site/pt/licoes/instalacao-linux/index.html,1,118
119,_site/pt/licoes/instalacao-windows/index.html,1,119
120,_site/pt/licoes/introducao-mysql-r/index.html,1,120
121,_site/pt/licoes/sumarizacao-narrativas-web-pyt...,1,121


In [63]:
merged_df = df.merge(file_counts_df, on='file', how='outer').sort_values(by="count_index", ascending=True)

In [64]:
merged_df[merged_df.file.str.contains("_site/assets", na=False)]

,file,line,message,count,count_index


In [30]:
import os
import re

EXTENSIONS = (".yml")

def replace_links_preserving_code_blocks(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Match code blocks (triple backticks) and inline code (`...`)
    code_blocks = list(re.finditer(r"(```.*?```|`[^`]*`)", content, re.DOTALL))
    modified = content
    offset = 0

    for match in code_blocks:
        start, end = match.span()
        segment = content[start:end]

        # Temporarily mark this section to skip
        placeholder = f"%%CODEBLOCK{start}%%"
        modified = modified[:start + offset] + placeholder + modified[end + offset:]
        offset += len(placeholder) - (end - start)

    # Replace all http:// with https://
    modified = re.sub(r"http://", "https://", modified)

    # Restore code blocks untouched
    for match in code_blocks:
        start = match.start()
        placeholder = f"%%CODEBLOCK{start}%%"
        modified = modified.replace(placeholder, match.group(0))

    if content != modified:
        print(f"✅ Updated: {file_path}")
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(modified)

def process_all_files(root="."):
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.endswith(EXTENSIONS) and "ph_authors" in fname:
                replace_links_preserving_code_blocks(os.path.join(dirpath, fname))

process_all_files()

✅ Updated: ./_data/ph_authors.yml
